In [2]:
import torch
from tokenizers import BertWordPieceTokenizer

In [8]:
import os

print(os.getcwd())

from pathlib import Path

print(Path.cwd())


c:\Data\Masai\Practice\Transformer\notebooks
c:\Data\Masai\Practice\Transformer\notebooks


In [20]:
from pathlib import Path

dir_path = Path('./../data')
# rglob("*") searches recursively
files = [f.name for f in dir_path.iterdir()]
print(files)

['download.ipynb', 'Gutenberg_Books+Metadata']


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(
    str(project_root)
)

from training.dataset import MLMDataset



In [3]:
dataset=MLMDataset('./../data/Gutenberg_Books+Metadata/processed/token_ids.pt',seq_len=10)

sample = dataset[1]
print (sample)
print(sample['input_ids'].dtype)
print(sample['labels'].dtype)

for i in range(5):
    sample= dataset[i]

    print(
        (sample['labels'] != -100).sum().item()
    )

{'input_ids': tensor([ 513, 7975,  833,  931,    4, 2021, 2867,  835,    4, 3359]), 'labels': tensor([-100, -100, -100, -100, 7480,  926, -100, -100,  825, -100])}
torch.int64
torch.int64
2
1
1
3
1


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from training.dataset import MLMDataset
from models.bert import MiniBERT
from models.mlm_head import MLMHead

class BERTForMLM(nn.Module):
    def __init__(self, tokenIDsFile, dmodel=256, noOfHeads=8, dffn=1024, noOfLayers=6, batchSize =32, seqLen=128, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.dataset=MLMDataset(tokenIDsFile, seqLen)
        self.dataLoader=DataLoader(self.dataset, batch_size=batchSize, shuffle=True)
        self.vocabSize=len(self.dataset.tokens)
        self.BERT= MiniBERT(self.vocabSize, seqLen, dmodel, noOfHeads, dffn, noOfLayers)
        self.MLMHead= MLMHead(dmodel, self.vocabSize)

    def forward(self):
        for batch in iter(self.dataLoader):
            input_ids = batch["input_ids"]
            labels = batch["labels"]
            X, att_matrices= self.BERT(input_ids)
            logits=self.MLMHead(X)
            

torch.int64
torch.int64
